# Synchronization Loss GPU A/B/C Benchmark

This notebook compares `SynchronizationLoss`, `SynchronizationLossFast`, and `SynchronizationLossPrefixSum` in `v1_model_utils/loss_functions.py`.

It checks:
- numerical equivalence (same seeds, same inputs)
- runtime (ms/call)
- optional GPU peak memory per call (when TensorFlow memory stats are available)


In [1]:
import contextlib
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import pickle as pkl

# Keep only GPU:0 visible to TensorFlow.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices(gpus[0], 'GPU')
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception:
        pass

# Ensure imports and relative data paths work regardless of notebook launch dir.
def _resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'v1_model_utils').exists() and (candidate / 'Synchronization_data').exists():
            return candidate
    raise RuntimeError('Could not locate project root containing v1_model_utils and Synchronization_data')

PROJECT_ROOT = _resolve_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from v1_model_utils import loss_functions as losses
from v1_model_utils import other_v1_utils

# chdir to v1_model_utils for imports to work correctly in models.py
# os.chdir(PROJECT_ROOT / 'v1_model_utils')
os.chdir(PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))


/tmp/ipykernel_86133/2293687584.py:8: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
2026-02-11 18:06:56.922378: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-11 18:06:56.960968: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-11 18:06

Project root: /home/jgalvan/Desktop/Neurocoding/V1_GLIF_model
TensorFlow: 2.15.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:4', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:5', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:6', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:7', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:8', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:9', device_type='GPU')]


2026-02-11 18:07:01.569772: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-02-11 18:07:01.572349: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-02-11 18:07:01.574955: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-02-11 18:07:01.577697: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX,

## Natural Scenes Generator Example

Use the new generator in `stim_dataset.py` to sample random Brain Observatory natural scenes and convert them to LGN inputs.

In [1]:
from pathlib import Path
import stim_dataset

# Load natural-scene templates from Allen Brain Observatory.
# # Returned array shape is [n_scenes, H, W] with pixel values in [0, 255].
# scenes = stim_dataset.load_via_allensdk(Path("./cache"))
# print(f"Loaded scenes shape: {scenes.shape}")
# print(f"Pixel range before resize: [{scenes.min()}, {scenes.max()}]")

# Build an infinite dataset that samples one random natural scene per example,
# resizes it to LGN dimensions, and converts it to LGN spike inputs.
natural_scene_ds = stim_dataset.generate_natural_scenes_stimulus(
    seq_len=200,
    pre_delay=50,
    post_delay=50,
    row_size=80,
    col_size=120,
    n_input=17400,
    data_dir='GLIF_network_nll_core',
    current_input=False,
    return_firing_rates=False,
    return_scene_id=True,
)

spikes_ns, scene_ids = next(iter(natural_scene_ds.batch(batch_size)))
print("Sample batch shape:", spikes_ns.shape)
print("Sample scene ids:", scene_ids.numpy())

# # Optional: run the benchmarked sync loss on a natural-scenes-driven batch.
# sync_nat = fast(tf.cast(spikes_ns, tf.float32), trim=True)
# print("Synchronization loss (natural scenes):", float(sync_nat.numpy()))


2026-02-11 18:41:34.824744: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-11 18:41:34.824805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-11 18:41:34.825850: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-11 18:41:34.832241: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-11 18:41:35.680910: W tensorflow/compiler/tf2

Loading natural scenes from .cache/natural_scenes.npy...
Found cached scenes, loading from disk...


2026-02-11 18:41:39.441096: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Computing spontaneous firing rates
Caching spontaneous firing rates
Computing temporal kernels


KeyboardInterrupt: 